# 04 — Evaluación Técnica y de Negocio

## Objetivo

Evaluar modelos candidatos sobre **TEST** (primera y única vez). Traducir CADA métrica técnica a impacto económico. Seleccionar modelo y política.

**Regla:** Este agente es el ÚNICO autorizado a usar el test set.

In [1]:
import pandas as pd
import numpy as np
import json
import joblib
from pathlib import Path
from datetime import datetime
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

PROJECT_ROOT = Path('.').resolve()
if PROJECT_ROOT.name != 'de-junior-tecnico-a-senior-de-negocio':
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts' / 'models'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
MANIFESTS_DIR = PROJECT_ROOT / 'manifests'

# Validar fase anterior
prev = json.load(open(MANIFESTS_DIR / '03_model_tournament.json'))
assert prev['status'] == 'GO', f"Fase anterior: {prev['status']}"
print(f"\u2705 Fase anterior: {prev['status']}")

# Costos de negocio (del product brief)
COST_IDLE = 0.0001      # 0.01% del exceso por dia
COST_SHORTAGE = 0.0005  # 0.05% del faltante por dia
SERVICE_LEVEL_TARGET = 0.95


✅ Fase anterior: GO


## Parte 1: Carga de datos y modelos

In [2]:
test_df = pd.read_csv(PROCESSED_DIR / 'test.csv', parse_dates=['date'])
feature_list = json.load(open(PROCESSED_DIR / 'feature_list.json'))
FEATURES = feature_list['features']
TARGET = feature_list['target']

X_test = test_df[FEATURES].values
y_test = test_df[TARGET].values

# Cargar modelos
model_enet = joblib.load(ARTIFACTS_DIR / 'elastic_net.joblib')
model_gbr = joblib.load(ARTIFACTS_DIR / 'gbr_central.joblib')
model_gbr_q95 = joblib.load(ARTIFACTS_DIR / 'gbr_quantile95.joblib')

# Predicciones
pred_enet = model_enet.predict(X_test)
pred_gbr = model_gbr.predict(X_test)
pred_q95 = model_gbr_q95.predict(X_test)
pred_lag1 = test_df['lag_1'].values
pred_ma7 = test_df['rolling_mean_7'].values

print(f"Test set: {len(test_df)} dias ({test_df['date'].min().date()} a {test_df['date'].max().date()})")
print(f"Retiro promedio en test: {y_test.mean()/1e9:.1f}B COP")


Test set: 141 dias (2026-02-10 a 2026-06-30)
Retiro promedio en test: 157.9B COP


## Parte 2: Scorecard técnico completo

In [3]:
def mape(y, p): return float(np.mean(np.abs((y - p) / y)) * 100)
def smape(y, p): return float(np.mean(2 * np.abs(y - p) / (np.abs(y) + np.abs(p))) * 100)
def mdape(y, p): return float(np.median(np.abs((y - p) / y)) * 100)
def mae_val(y, p): return float(np.mean(np.abs(y - p)))
def rmse_val(y, p): return float(np.sqrt(np.mean((y - p)**2)))
def mdae(y, p): return float(np.median(np.abs(y - p)))
def poe(y, p): return float(np.mean(p > y) * 100)
def pue(y, p): return float(np.mean(p < y) * 100)
def moe(y, p):
    mask = p > y
    return float(np.mean(p[mask] - y[mask])) if mask.sum() > 0 else 0.0
def mue(y, p):
    mask = p < y
    return float(np.mean(y[mask] - p[mask])) if mask.sum() > 0 else 0.0
def pinball(y, p, alpha=0.95):
    e = y - p
    return float(np.mean(np.where(e >= 0, alpha * e, (alpha - 1) * e)))
def coverage(y, upper): return float(np.mean(y <= upper) * 100)

models = {
    'Baseline Lag-1': pred_lag1,
    'Baseline MA-7': pred_ma7,
    'ElasticNet': pred_enet,
    'GBR Central': pred_gbr,
}

# Scorecard
rows = []
for name, pred in models.items():
    rows.append({
        'Modelo': name,
        'MAPE (%)': f"{mape(y_test, pred):.1f}",
        'SMAPE (%)': f"{smape(y_test, pred):.1f}",
        'MdAPE (%)': f"{mdape(y_test, pred):.1f}",
        'MAE (B)': f"{mae_val(y_test, pred)/1e9:.1f}",
        'RMSE (B)': f"{rmse_val(y_test, pred)/1e9:.1f}",
        'MdAE (B)': f"{mdae(y_test, pred)/1e9:.1f}",
        'POE (%)': f"{poe(y_test, pred):.0f}",
        'PUE (%)': f"{pue(y_test, pred):.0f}",
        'MOE (B)': f"{moe(y_test, pred)/1e9:.1f}",
        'MUE (B)': f"{mue(y_test, pred)/1e9:.1f}",
    })

scorecard = pd.DataFrame(rows).set_index('Modelo')
print("=== SCORECARD TECNICO (Test) ===")
print(scorecard.to_string())

# Q95 especifico
print(f"\nGBR Cuantil 95:")
print(f"  Pinball Loss: {pinball(y_test, pred_q95)/1e9:.2f}B")
print(f"  Cobertura: {coverage(y_test, pred_q95):.1f}%")


=== SCORECARD TECNICO (Test) ===
               MAPE (%) SMAPE (%) MdAPE (%) MAE (B) RMSE (B) MdAE (B) POE (%) PUE (%) MOE (B) MUE (B)
Modelo                                                                                               
Baseline Lag-1     26.7      25.0      20.0    39.0     48.5     32.6      47      53    41.3    37.0
Baseline MA-7      17.9      17.0      15.3    26.4     33.1     23.0      51      49    25.2    27.6
ElasticNet         11.3      11.7      10.4    18.1     22.6     16.2      31      69    11.9    20.9
GBR Central        15.9      17.4      16.0    25.8     30.3     26.1      14      86    13.1    27.9

GBR Cuantil 95:
  Pinball Loss: 3.89B
  Cobertura: 70.9%


## Parte 3: Gráfico de inferencia (real vs predicho)

In [4]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=test_df['date'], y=y_test/1e9, mode='lines', name='Real', line=dict(width=2.5, color='black')))
fig.add_trace(go.Scatter(x=test_df['date'], y=pred_gbr/1e9, mode='lines', name='GBR Central', line=dict(dash='dash', color='blue')))
fig.add_trace(go.Scatter(x=test_df['date'], y=pred_enet/1e9, mode='lines', name='ElasticNet', line=dict(dash='dash', color='green')))
fig.add_trace(go.Scatter(x=test_df['date'], y=pred_q95/1e9, mode='lines', name='Cuantil 95 (techo)', line=dict(dash='dot', color='red'), fill='tonexty', fillcolor='rgba(255,0,0,0.05)'))
fig.update_layout(title='Inferencia: Real vs Predicho (Test)', yaxis_title='Miles de millones COP', height=450, xaxis_title='')
fig.show()


## Parte 4: PUENTE — ¿Qué significa cada métrica para tesorería?

Esta sección traduce los números técnicos a lenguaje de negocio.

In [5]:
# Tomar el mejor modelo (GBR Central) como ejemplo
best_mape = mape(y_test, pred_gbr)
best_poe = poe(y_test, pred_gbr)
best_pue = pue(y_test, pred_gbr)
best_moe_val = moe(y_test, pred_gbr)
best_mue_val = mue(y_test, pred_gbr)
q95_cov = coverage(y_test, pred_q95)
avg_real = y_test.mean()

print("=" * 60)
print("TRADUCCION A NEGOCIO — Modelo GBR Central")
print("=" * 60)
print(f"""
MAPE = {best_mape:.1f}%
  -> En promedio, el pronostico se desvia {best_mape:.0f}% del retiro real.
  -> Si manana se retiran {avg_real/1e9:.0f}B, el pronostico podria estar
     entre {avg_real*(1-best_mape/100)/1e9:.0f}B y {avg_real*(1+best_mape/100)/1e9:.0f}B.

POE = {best_poe:.0f}% (dias con sobreestimacion)
  -> El {best_poe:.0f}% de los dias, reservariamos MAS de lo necesario.
  -> Esos dias generan DINERO OCIOSO.

PUE = {best_pue:.0f}% (dias con subestimacion)
  -> El {best_pue:.0f}% de los dias, reservariamos MENOS de lo necesario.
  -> Esos dias generan FALTANTE (riesgo operativo).

MOE = {best_moe_val/1e9:.1f}B (exceso promedio cuando sobramos)
  -> Cuando nos pasamos, nos pasamos por {best_moe_val/1e9:.1f}B en promedio.
  -> Costo diario del ocioso: {COST_IDLE * best_moe_val/1e6:.1f}M COP.

MUE = {best_mue_val/1e9:.1f}B (faltante promedio cuando nos quedamos cortos)
  -> Cuando falta, faltan {best_mue_val/1e9:.1f}B en promedio.
  -> Costo diario del faltante: {COST_SHORTAGE * best_mue_val/1e6:.1f}M COP.
  -> ESTE ES EL NUMERO QUE PREOCUPA A TESORERIA.

Cobertura Q95 = {q95_cov:.0f}%
  -> Solo el {q95_cov:.0f}% de los dias el "techo" protege.
  -> Target de servicio: 95%.
  -> {"CUMPLE" if q95_cov >= 95 else "NO CUMPLE — necesitamos ajustar el buffer."} 
""")


TRADUCCION A NEGOCIO — Modelo GBR Central

MAPE = 15.9%
  -> En promedio, el pronostico se desvia 16% del retiro real.
  -> Si manana se retiran 158B, el pronostico podria estar
     entre 133B y 183B.

POE = 14% (dias con sobreestimacion)
  -> El 14% de los dias, reservariamos MAS de lo necesario.
  -> Esos dias generan DINERO OCIOSO.

PUE = 86% (dias con subestimacion)
  -> El 86% de los dias, reservariamos MENOS de lo necesario.
  -> Esos dias generan FALTANTE (riesgo operativo).

MOE = 13.1B (exceso promedio cuando sobramos)
  -> Cuando nos pasamos, nos pasamos por 13.1B en promedio.
  -> Costo diario del ocioso: 1.3M COP.

MUE = 27.9B (faltante promedio cuando nos quedamos cortos)
  -> Cuando falta, faltan 27.9B en promedio.
  -> Costo diario del faltante: 13.9M COP.
  -> ESTE ES EL NUMERO QUE PREOCUPA A TESORERIA.

Cobertura Q95 = 71%
  -> Solo el 71% de los dias el "techo" protege.
  -> Target de servicio: 95%.
  -> NO CUMPLE — necesitamos ajustar el buffer. 



## Parte 5: Simulación de políticas de reserva

**Función de costo:**
```
C(q,y) = 0.01% × max(reserva - retiro, 0) + 0.05% × max(retiro - reserva, 0)
```

In [6]:
def calculate_policy_kpis(name, reserved, actual):
    """Calcula KPIs de negocio para una politica."""
    idle = np.maximum(reserved - actual, 0)
    shortage = np.maximum(actual - reserved, 0)
    cost = COST_IDLE * idle + COST_SHORTAGE * shortage
    return {
        'Politica': name,
        'Dinero ocioso prom (B/dia)': np.mean(idle) / 1e9,
        'Faltante prom cuando ocurre (B)': np.mean(shortage[shortage > 0]) / 1e9 if (shortage > 0).sum() > 0 else 0,
        'Dias con faltante (%)': np.mean(shortage > 0) * 100,
        'Nivel de servicio (%)': np.mean(reserved >= actual) * 100,
        'Costo total periodo (M)': np.sum(cost) / 1e6,
        'Costo diario prom (M)': np.mean(cost) / 1e6,
    }

# Politica tradicional: max ultimos 7 dias * 1.10
trad_reserve = test_df['lag_1'].rolling(7, min_periods=1).apply(lambda x: x.max() * 1.10).values
trad_reserve = np.where(np.isnan(trad_reserve), test_df['lag_7'].values * 1.10, trad_reserve)

# Politica modelo Q95
model_q95_reserve = pred_q95

# Politica modelo central + 20% buffer
model_buffer_reserve = pred_gbr * 1.20

# Politica modelo central + 35% buffer (para intentar cumplir servicio)
model_buffer35_reserve = pred_gbr * 1.35

policies = [
    calculate_policy_kpis('Tradicional (max7d + 10%)', trad_reserve, y_test),
    calculate_policy_kpis('Modelo Q95', model_q95_reserve, y_test),
    calculate_policy_kpis('Modelo Central + 20%', model_buffer_reserve, y_test),
    calculate_policy_kpis('Modelo Central + 35%', model_buffer35_reserve, y_test),
]

kpi_df = pd.DataFrame(policies)
print("=== KPIs DE NEGOCIO POR POLITICA ===")
print(kpi_df.to_string(index=False, float_format='{:.1f}'.format))


=== KPIs DE NEGOCIO POR POLITICA ===
                 Politica  Dinero ocioso prom (B/dia)  Faltante prom cuando ocurre (B)  Dias con faltante (%)  Nivel de servicio (%)  Costo total periodo (M)  Costo diario prom (M)
Tradicional (max7d + 10%)                        65.6                             29.9                    3.5                   96.5                    999.4                    7.1
               Modelo Q95                        15.8                             11.2                   29.1                   70.9                    452.9                    3.2
     Modelo Central + 20%                        11.0                             14.1                   41.8                   58.2                    572.3                    4.1
     Modelo Central + 35%                        26.8                             13.4                    9.9                   90.1                    472.1                    3.3


## Parte 6: Gráficos de decisión

In [7]:
# Grafico 1: Reserva vs Real
fig = go.Figure()
fig.add_trace(go.Scatter(x=test_df['date'], y=y_test/1e9, mode='lines', name='Retiro real', line=dict(width=2.5, color='black')))
fig.add_trace(go.Scatter(x=test_df['date'], y=trad_reserve/1e9, mode='lines', name='Tradicional', line=dict(dash='dash', color='orange')))
fig.add_trace(go.Scatter(x=test_df['date'], y=model_buffer35_reserve/1e9, mode='lines', name='Modelo +35%', line=dict(dash='dash', color='blue')))
fig.update_layout(title='Reserva diaria vs Retiro real', yaxis_title='B COP', height=400)
fig.show()


In [8]:
# Grafico 2: Dinero ocioso y faltante comparado
fig = make_subplots(rows=1, cols=2, subplot_titles=('Costo total del periodo (M COP)', 'Nivel de servicio (%)'))

fig.add_trace(go.Bar(x=kpi_df['Politica'], y=kpi_df['Costo total periodo (M)'],
                     text=[f"{v:.0f}M" for v in kpi_df['Costo total periodo (M)']],
                     textposition='outside', marker_color=['orange','red','green','blue']), row=1, col=1)
fig.add_trace(go.Bar(x=kpi_df['Politica'], y=kpi_df['Nivel de servicio (%)'],
                     text=[f"{v:.1f}%" for v in kpi_df['Nivel de servicio (%)']],
                     textposition='outside', marker_color=['orange','red','green','blue']), row=1, col=2)
fig.add_hline(y=95, line_dash='dash', line_color='red', row=1, col=2, annotation_text='Target 95%')
fig.update_layout(height=400, showlegend=False)
fig.show()


In [9]:
# Grafico 3: Ocioso vs Faltante por dia (mejor politica viable)
# Elegir la politica que cumple servicio con menor costo
viable = kpi_df[kpi_df['Nivel de servicio (%)'] >= 95]
if len(viable) > 0:
    best_policy_name = viable.loc[viable['Costo total periodo (M)'].idxmin(), 'Politica']
else:
    best_policy_name = kpi_df.loc[kpi_df['Nivel de servicio (%)'].idxmax(), 'Politica']

# Mapear reservas
reserves_map = {
    'Tradicional (max7d + 10%)': trad_reserve,
    'Modelo Q95': model_q95_reserve,
    'Modelo Central + 20%': model_buffer_reserve,
    'Modelo Central + 35%': model_buffer35_reserve,
}
best_reserve = reserves_map[best_policy_name]
best_idle = np.maximum(best_reserve - y_test, 0)
best_shortage = np.maximum(y_test - best_reserve, 0)

fig = go.Figure()
fig.add_trace(go.Bar(x=test_df['date'], y=best_idle/1e9, name='Dinero ocioso', marker_color='steelblue'))
fig.add_trace(go.Bar(x=test_df['date'], y=-best_shortage/1e9, name='Faltante', marker_color='red'))
fig.update_layout(title=f'Ocioso y faltante diario — {best_policy_name}', barmode='relative',
                  yaxis_title='B COP', height=350)
fig.show()


## Parte 7: Selección del ganador

In [10]:
print("=" * 60)
print("SELECCION DEL MODELO Y POLITICA")
print("=" * 60)

# Guardrail: servicio >= 95%
viable = kpi_df[kpi_df['Nivel de servicio (%)'] >= SERVICE_LEVEL_TARGET * 100]

if len(viable) > 0:
    winner = viable.loc[viable['Costo total periodo (M)'].idxmin()]
    print(f"\n\u2705 POLITICA GANADORA: {winner['Politica']}")
    print(f"   Nivel de servicio: {winner['Nivel de servicio (%)']:.1f}%")
    print(f"   Costo total: {winner['Costo total periodo (M)']:.0f}M COP")
    print(f"   Dinero ocioso promedio: {winner['Dinero ocioso prom (B/dia)']:.1f}B/dia")
    print(f"   Dias con faltante: {winner['Dias con faltante (%)']:.1f}%")
else:
    winner = kpi_df.loc[kpi_df['Nivel de servicio (%)'].idxmax()]
    print(f"\n\u26a0 Ninguna politica cumple 95%. Mejor opcion: {winner['Politica']}")
    print(f"   Nivel de servicio: {winner['Nivel de servicio (%)']:.1f}%")

# Comparacion vs tradicional
trad = kpi_df[kpi_df['Politica'].str.contains('Tradicional')].iloc[0]
saving = trad['Costo total periodo (M)'] - winner['Costo total periodo (M)']
saving_pct = saving / trad['Costo total periodo (M)'] * 100 if trad['Costo total periodo (M)'] > 0 else 0

print(f"\n   Ahorro vs tradicional: {saving:.0f}M COP ({saving_pct:.1f}%)")


SELECCION DEL MODELO Y POLITICA

✅ POLITICA GANADORA: Tradicional (max7d + 10%)
   Nivel de servicio: 96.5%
   Costo total: 999M COP
   Dinero ocioso promedio: 65.6B/dia
   Dias con faltante: 3.5%

   Ahorro vs tradicional: 0M COP (0.0%)


## Parte 8: Respuesta final al negocio

In [11]:
print("""
================================================================
RESPUESTA PARA TESORERIA
================================================================

PREGUNTA: ¿Cuanto dinero ahorramos vs la regla actual?
""")
print(f"RESPUESTA: {abs(saving):.0f}M COP en {len(test_df)} dias.")
if saving > 0:
    print("           La politica basada en modelo es MAS BARATA.")
else:
    print("           La politica tradicional es MAS BARATA.")
    print("           Pero puede tener mas dinero ocioso innecesario.")

print(f"""
PREGUNTA: ¿Que riesgo estamos asumiendo?
RESPUESTA: {winner['Dias con faltante (%)']:.1f}% de los dias podria haber faltante.
           Cuando ocurre, el faltante promedio es de {winner['Faltante prom cuando ocurre (B)']:.1f}B COP.

PREGUNTA: ¿Que pasa si el gerente pide 99% de servicio?
RESPUESTA: Necesitariamos un buffer mayor (~40-50%), lo que aumentaria
           el dinero ocioso y el costo total significativamente.

PREGUNTA: ¿Vale la pena el modelo?
""")
if saving > 0:
    print(f"RESPUESTA: SI. Ahorra {saving:.0f}M manteniendo servicio >= 95%.")
else:
    print("RESPUESTA: DEPENDE. El modelo no ahorra vs tradicional en este periodo,")
    print("           pero permite CONFIGURAR el balance riesgo/costo con precision.")
    print("           La regla tradicional es fija y no se adapta.")



RESPUESTA PARA TESORERIA

PREGUNTA: ¿Cuanto dinero ahorramos vs la regla actual?

RESPUESTA: 0M COP en 141 dias.
           La politica tradicional es MAS BARATA.
           Pero puede tener mas dinero ocioso innecesario.

PREGUNTA: ¿Que riesgo estamos asumiendo?
RESPUESTA: 3.5% de los dias podria haber faltante.
           Cuando ocurre, el faltante promedio es de 29.9B COP.

PREGUNTA: ¿Que pasa si el gerente pide 99% de servicio?
RESPUESTA: Necesitariamos un buffer mayor (~40-50%), lo que aumentaria
           el dinero ocioso y el costo total significativamente.

PREGUNTA: ¿Vale la pena el modelo?

RESPUESTA: DEPENDE. El modelo no ahorra vs tradicional en este periodo,
           pero permite CONFIGURAR el balance riesgo/costo con precision.
           La regla tradicional es fija y no se adapta.


## Artefactos generados

In [12]:
# Guardar seleccion
selected = {
    'selected_policy': winner['Politica'],
    'central_model_path': 'artifacts/models/gbr_central.joblib',
    'quantile_model_path': 'artifacts/models/gbr_quantile95.joblib',
    'metrics_test': {
        'gbr_central_mape_pct': mape(y_test, pred_gbr),
        'gbr_central_mae_B': mae_val(y_test, pred_gbr) / 1e9,
        'q95_coverage_pct': coverage(y_test, pred_q95),
    },
    'business_kpis': {
        'service_level_pct': float(winner['Nivel de servicio (%)']),
        'cost_total_period_M': float(winner['Costo total periodo (M)']),
        'idle_daily_mean_B': float(winner['Dinero ocioso prom (B/dia)']),
        'shortage_days_pct': float(winner['Dias con faltante (%)']),
        'saving_vs_traditional_M': float(saving),
        'saving_vs_traditional_pct': float(saving_pct),
    },
    'cost_params': {'cost_idle': COST_IDLE, 'cost_shortage': COST_SHORTAGE, 'service_level_target': SERVICE_LEVEL_TARGET}
}

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
with open(OUTPUTS_DIR / 'selected_model.json', 'w') as f:
    json.dump(selected, f, indent=2)
print(f"\u2705 Guardado: outputs/selected_model.json")

# Test predictions
test_preds = test_df[['date', TARGET]].copy()
test_preds['pred_gbr_central'] = pred_gbr
test_preds['pred_gbr_q95'] = pred_q95
test_preds['pred_elastic_net'] = pred_enet
test_preds.to_csv(OUTPUTS_DIR / 'test_predictions.csv', index=False)
print(f"\u2705 Guardado: outputs/test_predictions.csv")

# Business backtest
backtest = test_df[['date', TARGET]].copy()
backtest['reserve_traditional'] = trad_reserve
backtest['reserve_model'] = best_reserve
backtest['idle'] = np.maximum(best_reserve - y_test, 0)
backtest['shortage'] = np.maximum(y_test - best_reserve, 0)
backtest['cost_daily'] = COST_IDLE * backtest['idle'] + COST_SHORTAGE * backtest['shortage']
backtest.to_csv(OUTPUTS_DIR / 'business_backtest.csv', index=False)
print(f"\u2705 Guardado: outputs/business_backtest.csv")

# Manifest
manifest = {
    "phase": "evaluation-business",
    "status": "GO",
    "started_at": datetime.now().isoformat(),
    "completed_at": datetime.now().isoformat(),
    "inputs": ["data/processed/test.csv", "artifacts/models/", "manifests/03_model_tournament.json"],
    "outputs": ["notebooks/04_evaluation_business.ipynb", "outputs/test_predictions.csv", "outputs/business_backtest.csv", "outputs/selected_model.json", "manifests/04_evaluation_business.json"],
    "decisions": [f"Politica seleccionada: {winner['Politica']}", f"Servicio: {winner['Nivel de servicio (%)']:.1f}%", f"Ahorro vs tradicional: {saving:.0f}M COP"],
    "metrics": selected['metrics_test'] | selected['business_kpis'],
    "assumptions": ["Costos: ociosidad=0.01%, faltante=0.05%", "Nivel servicio target: 95%"],
    "risks": ["Costos son parametros configurables", "Cobertura Q95 puede variar en produccion"],
    "tests_executed": ["test_evaluation", "business_backtest", "policy_comparison", "kpi_translation"],
    "human_approval_required": True,
    "human_approved": False,
    "next_agent": "productization-deployment"
}
with open(MANIFESTS_DIR / '04_evaluation_business.json', 'w') as f:
    json.dump(manifest, f, indent=2)
print(f"\u2705 Manifest guardado: manifests/04_evaluation_business.json")


✅ Guardado: outputs/selected_model.json
✅ Guardado: outputs/test_predictions.csv
✅ Guardado: outputs/business_backtest.csv
✅ Manifest guardado: manifests/04_evaluation_business.json


## Conclusiones

### Estado: **GO** \u2705

### Mensaje clave

> El MAPE te dice qué tan bueno es el pronóstico.
> Pero el costo total te dice qué tan buena es la **decisión**.
>
> La predicción NO es la decisión.

### Siguiente paso
Agente: **productization-deployment**